# Temporal Train/Test Split & Feature Scaling Pipeline

**Objective**: Prepare the final SARS-CoV-2 dataset for supervised learning by creating a temporally ordered train/test split and standardizing numeric features for model training.

***

## 1. Setup & Imports

**Imports for data manipulation, serialization, and feature scaling.**

In [9]:
import pandas as pd
import joblib
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')



## 2. Load Processed Dataset
**Loads the ML-ready dataset (3,500 sequences, balanced across 7 variants) containing temporal features, variant metadata, and spike mutation counts as the regression target.**

In [3]:
# Load dataset
df = pd.read_csv('../data/processed/final_dataset.csv')
print(f"Dataset loaded: {df.shape}")

Dataset loaded: (3500, 12)


## 3. Temporal Train/Test Split (80/20)
**Creates a realistic temporal split: trains on earlier pandemic data (2020-2022) and tests on emerging variants (2023+, xbb15-heavy). This prevents temporal leakage and evaluates generalization to future sequences.**



In [4]:
# TEMPORAL SPLIT (80% earliest data)
df_sorted = df.sort_values('days_since_start')
cutoff_date = df_sorted['days_since_start'].quantile(0.8)
train_mask = df_sorted['days_since_start'] <= cutoff_date

X_train = df.loc[train_mask].drop(['phenotype_number spike muts from Wuhan-Hu-1', 'accession'], axis=1)
y_train = df.loc[train_mask, 'phenotype_number spike muts from Wuhan-Hu-1']
X_test = df.loc[~train_mask].drop(['phenotype_number spike muts from Wuhan-Hu-1', 'accession'], axis=1)
y_test = df.loc[~train_mask, 'phenotype_number spike muts from Wuhan-Hu-1']

print(f"Train: {len(X_train)} ({train_mask.mean():.1%}) | Test: {len(X_test)}")
print("\nTrain by year:\n", df.loc[train_mask, 'year'].value_counts().sort_index())
print("\nTrain by variant:\n", df.loc[train_mask, '_source_variant'].value_counts())
print("\nTest by variant:\n", df.loc[~train_mask, '_source_variant'].value_counts())


Train: 2808 (80.2%) | Test: 692

Train by year:
 year
2020.0     536
2021.0    1324
2022.0     948
Name: count, dtype: int64

Train by variant:
 _source_variant
b1             500
delta          500
omicron_ba1    499
beta           499
omicron        498
ba2            312
Name: count, dtype: int64

Test by variant:
 _source_variant
xbb15          500
ba2            188
omicron          2
beta             1
omicron_ba1      1
Name: count, dtype: int64


## 4. Numeric Feature Standardization

**Standardizes only numeric temporal features (`year`, `month`, `days_since_start`) using StandardScaler fit on train data. Categorical features (variants) preserved as-is for tree-based models.**

In [5]:
# SCALE NUMERIC FEATURES 
numeric_cols = X_train.select_dtypes(include=['number']).columns
scaler = StandardScaler().fit(X_train[numeric_cols])

X_train_scaled = pd.DataFrame(
    scaler.transform(X_train[numeric_cols]), 
    columns=numeric_cols, 
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test[numeric_cols]), 
    columns=numeric_cols, 
    index=X_test.index
)


## 5. Save Preprocessed Splits

**Serializes the complete preprocessing pipeline artifacts. The `ml_splits_scaled.pkl` file is the single input for the next step.**

In [7]:
# SAVE SPLITS
joblib.dump({
    'X_train': X_train_scaled,
    'y_train': y_train,
    'X_test': X_test_scaled,
    'y_test': y_test,
    'scaler': scaler,
    'numeric_cols': numeric_cols
}, '../data/processed/ml_splits_scaled.pkl')

print(f"X_train shape: {X_train_scaled.shape}")
print(f"Target range: train {y_train.min():.1f}-{y_train.max():.1f}, test {y_test.min():.1f}-{y_test.max():.1f}")

X_train shape: (2808, 4)
Target range: train 1.0-36.0, test 10.0-42.0


# Machine Learning models 

## Model 1: XGBoost (Gradient Boosting)
XGBoost: Gradient boosting ensemble. Captures non-linear interactions and feature importance natively. Expected R² 0.70-0.80.

In [12]:
# Initialize XGBoost regressor
xgb_model = XGBRegressor(
    n_estimators=100,           # Number of boosting rounds
    max_depth=3,                # Tree depth (prevent overfitting)
    learning_rate=0.05,          # Shrinkage (regularization)
    subsample=0.7,              # Fraction of samples per tree
    colsample_bytree=0.6,
    reg_alpha=1.0,          
    reg_lambda=2.0,       
    verbosity=0
)

drop_cols = ['phenotype_number spike muts from Wuhan-Hu-1', 'accession', 
             'virus_name', 'geographic_location', 'isolate_collection_date', 
             'phenotype_clade', 'phenotype_clade growth', 'collection_date', 
             '_source_variant']  

X_train = df.loc[train_mask].drop(drop_cols, axis=1, errors='ignore')
y_train = df.loc[train_mask, 'phenotype_number spike muts from Wuhan-Hu-1']
X_test = df.loc[~train_mask].drop(drop_cols, axis=1, errors='ignore')
y_test = df.loc[~train_mask, 'phenotype_number spike muts from Wuhan-Hu-1']


# Train
print("Training XGBoost...")
xgb_model.fit(X_train, y_train, verbose=False)

# Predict
xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred = xgb_model.predict(X_test)

# Evaluate
xgb_r2_train = r2_score(y_train, xgb_train_pred)
xgb_r2_test = r2_score(y_test, xgb_test_pred)
xgb_mae_test = mean_absolute_error(y_test, xgb_test_pred)
xgb_rmse_test = np.sqrt(mean_squared_error(y_test, xgb_test_pred))

print(f"XGBoost Results:")
print(f"  Train R²: {xgb_r2_train:.3f}")
print(f"  Test R²:  {xgb_r2_test:.3f}")
print(f"  Test MAE: {xgb_mae_test:.3f}")
print(f"  Test RMSE: {xgb_rmse_test:.3f}")


Training XGBoost...
XGBoost Results:
  Train R²: 0.919
  Test R²:  -2.627
  Test MAE: 8.508
  Test RMSE: 9.579


## Overfitting Diagnosis and Data Leakage Analysis

Initial model evaluation revealed a severe discrepancy between training and test performance:

* **Training R² ≈ 0.90**
* **Test R² ≈ −2.5**

This pattern is a classic signature of **overfitting caused by distribution shift**, not merely model complexity. The issue arises from **temporal and lineage leakage**, where the model is learning time-related patterns instead of biologically meaningful genomic signals.

### Root Cause

The dataset split unintentionally created a strong separation between **early-pandemic** and **late-emerging variants**:

| Set              | Composition                        | Biological Profile                   |
| ---------------- | ---------------------------------- | ------------------------------------ |
| **Training set** | Mostly variants from **2020–2022** | Low spike mutation counts            |
| **Test set**     | ~72% **XBB.1.5 (2023+)**           | Naturally high spike mutation counts |

Because temporal features (year, month, days since pandemic start) were included, the model learned a shortcut:

> **“Later time → more mutations”**

Instead of learning relationships between **specific mutations** and the target phenotype, the model exploited time as a proxy for evolutionary progression. When evaluated on a test set dominated by a recent lineage (XBB.1.5), predictions failed catastrophically because the learned temporal rule does not generalize across variant evolution.


In [15]:
# 1. STRATIFIED CV BY VARIANT
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.preprocessing import LabelEncoder

# Re-add variant grouping
df['variant_group'] = df['_source_variant']
le = LabelEncoder()
groups = le.fit_transform(df.loc[X_train.index, 'variant_group'])

xgb_cv = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, reg_lambda=2, random_state=42)
scores = cross_val_score(xgb_cv, X_train, y_train, cv=GroupKFold(n_splits=5), groups=groups, scoring='r2')
print(f"5-fold CV R² by variant: {scores.mean():.3f} ± {scores.std():.3f}")

# 2. NO TEMPORAL FEATURES
X_train_no_time = X_train.drop(['year', 'month', 'days_since_start'], axis=1, errors='ignore')
X_test_no_time = X_test.drop(['year', 'month', 'days_since_start'], axis=1, errors='ignore')

xgb_notime = XGBRegressor(n_estimators=100, max_depth=3, random_state=42)
xgb_notime.fit(X_train_no_time, y_train)
pred_notime = xgb_notime.predict(X_test_no_time)
print(f"No time features - Test R²: {r2_score(y_test, pred_notime):.3f}")


5-fold CV R² by variant: 0.000 ± 0.000


IndexError: list index out of range

In [16]:
print("X_train_no_time shape:", X_train_no_time.shape)
print("X_train_no_time columns:", X_train_no_time.columns.tolist())

X_train_no_time shape: (2808, 0)
X_train_no_time columns: []


## **Model Performance Interpretation and Methodological Reassessment**

The initial modeling results show a strong discrepancy between training and test performance:

| Model   | Train R² | Test R²   | MAE  | Interpretation                |
| ------- | -------- | --------- | ---- | ----------------------------- |
| XGBoost | 0.919    | **−2.63** | 8.51 | Severe generalization failure |

Such a pattern indicates **model overfitting**, but not in the usual sense of excessive model complexity. Instead, the failure originates from **dataset structure and information leakage**.

---

### **Diagnostic Findings**

#### 1. Absence of True Biological Predictors

When temporal variables are removed (year, month, days since pandemic start), the feature matrix becomes effectively empty in terms of biological signal. The remaining predictors are mostly metadata proxies rather than mutation-specific representations. This confirms that the model is not learning mutation–phenotype relationships.

#### 2. Temporal and Lineage Leakage

The data split produced a strong distribution shift:

| Set              | Composition                    | Mutation Profile            |
| ---------------- | ------------------------------ | --------------------------- |
| **Training set** | Mostly variants from 2020–2022 | Lower spike mutation counts |
| **Test set**     | ~72% XBB.1.5 (2023+)           | Higher mutation counts      |

Because temporal features were included, the model learned a shortcut rule:

> **Later time → more mutations**

This is an evolutionary trend, not a mechanistic genomic relationship. The model therefore performs well on training data but collapses when predicting a lineage that represents a new evolutionary regime.

---

### **EDA Consistency**

Exploratory analysis previously showed:

* Spike mutation count acts as a **fitness proxy**, not direct fitness
* Later variants cluster separately in feature space
* Temporal progression strongly correlates with mutation accumulation

The modeling outcome is therefore consistent with the EDA findings: **temporal signal dominates engineered features**, masking true biological effects.

---

## **Conclusion of the Diagnostic Phase**

The current feature set does not contain sufficient **mutation-level biological representation**. Model performance was driven primarily by pandemic chronology rather than genomic mechanisms. Continuing with classical ML on these features would produce misleading conclusions.

---

## **Next Step: Transition to Protein Language Models (PLMs)**

To capture meaningful biological signal, the project will pivot to **sequence-based representation learning**.

### Rationale

Protein Language Models such as **ESM-2** encode structural, evolutionary, and epistatic relationships directly from amino acid sequences. Unlike simple mutation counts, these embeddings represent how combinations of mutations affect protein function.


